# FACULTY SOLUTION — Week 10: The cycle turns — where the consequences converge
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['product_elasticities', 'refinery_yields', 'recession_params', 'binding_rules', 'team_prior_state']}

In [ ]:

def compute(d):
    el = d['product_elasticities'].set_index('product')['income_elasticity']; g = d['recession_params']['value'].iloc[0]
    yl = d['refinery_yields'].set_index('refinery'); b = {x.parameter: x.value for x in d['binding_rules'].itertuples()}
    r = {}
    for p in el.index: r['demand_hit_' + p] = el[p] * g
    r['blended_demand_hit'] = sum(el[p] * g * s for p, s in zip(d['product_elasticities']['product'], d['product_elasticities']['demand_share_blended']))
    for rf, key in [('Baton Rouge', 'br'), ('Rotterdam', 'rot'), ('Singapore', 'sing')]:
        r['refinery_hit_' + key] = sum(yl.loc[rf, p] * el[p] * g for p in ['gasoline', 'diesel', 'jet'])
    r['hardest_hit_refinery'] = min(['Baton Rouge', 'Rotterdam', 'Singapore'], key=lambda x: r['refinery_hit_' + {'Baton Rouge': 'br', 'Rotterdam': 'rot', 'Singapore': 'sing'}[x]])
    for t in d['team_prior_state'].itertuples():
        k = t.team.replace('reference_', '')
        flags = {'capex': t.cancellable_capex_musd < b['min_cancellable_capex_musd'], 'hedge': t.crude_hedge_coverage < b['min_crude_hedge_coverage'],
                 'delacroix_cover': t.br_reported_margin_strong == 1, 'straits_pacific': t.straits_pacific_standing in ('strained', 'hostile'),
                 'cash': t.cash_cushion_musd < b['min_cash_cushion_musd']}
        for f, v in flags.items(): r[f'{k}_{f}'] = 'binding' if v else 'not binding'
        r[f'{k}_binding_count'] = sum(flags.values())
    return r


In [ ]:
fx = json.load(open('../fixtures/week10_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')